### 1. Simple Self-Attention+scaled dot Implementation Code (PyTorch)

In [6]:

import torch
import torch.nn.functional as F   # torch.nn.functional (usually imported as F) is a module that contains functions for neural networks.

# example input(static embedding): 3 words, embedding size = 4// i love cats  
X = torch.tensor([
    [1.0, 0.0, 1.0, 0.0],     # Word 1: "I"
    [0.0, 2.0, 0.0, 2.0],     # Word 2: "love"

    [1.0, 1.0, 1.0, 1.0]      # Word 3: "cats"
])

# in big dataset, at first nn.embedding creates for every word vector of d dimensions.embedding_layer = nn.Embedding(vocab_size, embed_dim)

d = X.shape[1]      # embedding dimension We extract the embedding dimension (4) for scaling later
                    #shape[1] gives the second dimension (columns)


# initialize weight matrices
Wq = torch.rand(d, d)        # We create three random matrices for Query, Key, Value
                             #In real transformers: These are learned during training
Wk = torch.rand(d, d)
Wv = torch.rand(d, d)

# compute Q, K, V
Q = X @ Wq                # We multiply our input X with each weight matrix
                          #The @ symbol is matrix multiplication (dot product)
                          #Each word gets its own Query, Key, and Value vector
K = X @ Wk
V = X @ Wv

print("\n Query vector is:")

print(Q)




# attention scores
scores = Q @ K.T          # Result is a [3, 3] matrix showing relationships between all words.Dot product measures similarity. If Q[i] and K[j] point in similar directions, they have a high score.

                          #scores[0, 1] = How much should Word 1 ("I") pay attention to Word 2 ("love")?
                          #scores[2, 0] = How much should Word 3 ("cats") pay attention to Word 1 ("I")?

# scale scores
scores = scores / (d ** 0.5)       # To prevent extremely small gradients during training
                                   # Without scaling, dot products can get very large, pushing softmax into regions with tiny gradients.Scaling solve high variance problem.

# softmax
attention_weights = F.softmax(scores, dim=1)    # [3,3] Softmax converts scores into probabilities that sum to 1
                                                # dim=1 means we apply softmax across columns (for each row)

# final output
output = attention_weights @ V         # Each word's final representation now contains: Its own information (from its Value)
                                       # PLUS context from other words (weighted by attention)  
          
print("Attention Weights:\n", attention_weights)
print("Output:\n", output)


 Query vector is:
tensor([[1.2483, 1.5069, 0.4373, 1.1590],
        [2.8018, 1.9582, 2.0675, 2.3605],
        [2.6492, 2.4860, 1.4711, 2.3392]])
Attention Weights:
 tensor([[0.0283, 0.5378, 0.4339],
        [0.0022, 0.5461, 0.4517],
        [0.0019, 0.5756, 0.4225]])
Output:
 tensor([[2.5253, 2.0021, 1.7669, 1.6995],
        [2.5645, 2.0222, 1.7899, 1.7200],
        [2.5766, 2.0093, 1.7892, 1.7162]])


### 2.Multi-head attention Code

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F


d_model = 8                      # Total embedding dimension (each word represented by 8 numbers)
num_heads = 4                     # Number of parallel attention heads
head_dim = d_model // num_heads   # 4 - Each head works with 4 features

# ==================== INPUT ====================
# Input: batch=1, sequence=3 words, d_model=8

x = torch.rand(1, 3, 8)           # [batch_size, seq_len, d_model] - 3 random words

print("\n Input vector(3,8)-3 words,8 dim: ")
print(x)                          # Shows 3 words, each with 8 random numbers

# STEP 1: LINEAR PROJECTIONS 

# Create learnable weight matrices for Query, Key, Value
Wq = nn.Linear(d_model, d_model)  # 8→8 - transforms input to QUERY vectors (asks "what to look for?")
Wk = nn.Linear(d_model, d_model)  # 8→8 - transforms input to KEY vectors (says "here's my label")
Wv = nn.Linear(d_model, d_model)  # 8→8 - transforms input to VALUE vectors (provides actual content)

print("\n weight for query: ")
print(Wq)                         # Shows layer info, not matrix (Linear(in_features=8, out_features=8))
print(Wq.weight)                   # Actual 8×8 weight matrix (randomly initialized)

# Apply projections to input
Q = Wq(x)                          # [1,3,8] - Query vectors: what each word is searching for
K = Wk(x)                          # [1,3,8] - Key vectors: how each word can be found
V = Wv(x)                          # [1,3,8] - Value vectors: actual information to pass

print("\n Before splitting Query:")

print(Q)

# STEP 2: RESHAPE FOR MULTI-HEAD 
# Split the 8 features into 2 heads of 4 features each
# view() reshapes without changing data: [batch, words, features] → [batch, words, heads, head_dim]
# transpose(1,2) swaps words and heads: [batch, heads, words, head_dim] - heads first for parallel processing
Q = Q.view(1, 3, num_heads, head_dim).transpose(1, 2)  # [1,2,3,4] - Head1 gets words1-3 with first4 features, Head2 gets last4
K = K.view(1, 3, num_heads, head_dim).transpose(1, 2)  # [1,2,3,4] - Same reshaping for Keys
V = V.view(1, 3, num_heads, head_dim).transpose(1, 2)  # [1,2,3,4] - Same reshaping for Values

print("\n After splitting Query:")
print(Q)                          # Shows: [head1: all 3 words×4features, head2: all3 words×4features]


# STEP 3: SCALED DOT-PRODUCT ATTENTION 
# Compute attention scores: Q × K.T gives similarity between words
# transpose(-2,-1) swaps last two dims: [1,2,3,4] → [1,2,4,3] for matrix multiplication
# / √head_dim (√4=2) prevents large values that kill gradients
scores = torch.matmul(Q, K.transpose(-2, -1)) / (head_dim ** 0.5)  # [1,2,3,3] - attention scores for each head


# Convert scores to probabilities (0 to 1, sum to 1)
attn_weights = F.softmax(scores, dim=-1)  # [1,2,3,3] - attention weights: how much each word attends to others


# Apply attention weights to values: weighted sum of values
head_output = torch.matmul(attn_weights, V)  # [1,2,3,4] - each word now contains info from other words

print("\n 2 heads combined in tensor:")
print(head_output)                 # [batch, heads, words, head_dim] - contextualized per head

# ==================== STEP 4: CONCATENATE HEADS BACK ====================
# transpose(1,2): [batch,heads,words,dim] → [batch,words,heads,dim] - group by word first
# contiguous(): ensures memory is continuous after transpose scattered value will be organized
# view(1,3,8): [batch,words,heads,dim] → [batch,words,heads×dim] - flatten heads and features
concat = head_output.transpose(1, 2).contiguous().view(1, 3, d_model)  # [1,3,8] - back to original shape

print("\n After 2 heads concatenated: ")
print(concat)                      # Each word: [head1_4features + head2_4features] = 8 features

# ==================== STEP 5: FINAL LINEAR PROJECTION ====================
# Create learnable layer to blend information from both heads
Wo = nn.Linear(d_model, d_model)   # 8→8 - learns optimal combination of head outputs
output = Wo(concat)                 # [1,3,8] - final blended representation

# ==================== OUTPUT ====================
print("Input shape:", x.shape)      # [1,3,8] - original
print("Output shape:", output.shape) # [1,3,8] - same shape, but values now contain context from all words!


 Input vector(3,8)-3 words,8 dim: 
tensor([[[0.7383, 0.3260, 0.5236, 0.0262, 0.6537, 0.0582, 0.3749, 0.6910],
         [0.8182, 0.4915, 0.6809, 0.0640, 0.0301, 0.8785, 0.7172, 0.3471],
         [0.1740, 0.4097, 0.0951, 0.8453, 0.4874, 0.5468, 0.7350, 0.6845]]])

 weight for query: 
Linear(in_features=8, out_features=8, bias=True)
Parameter containing:
tensor([[-0.1409, -0.1664, -0.2830, -0.0356, -0.2420,  0.2915,  0.0868, -0.3132],
        [ 0.0896, -0.2075,  0.2676,  0.0945, -0.2471,  0.0471, -0.1421, -0.2744],
        [ 0.1455, -0.0541, -0.3207, -0.0836,  0.1918, -0.2876, -0.2759,  0.1116],
        [ 0.2878, -0.2976,  0.2586, -0.2649, -0.0161, -0.0134,  0.3127, -0.0421],
        [ 0.3297,  0.2694,  0.3128,  0.0239,  0.3509, -0.2645, -0.1925, -0.2289],
        [-0.2599, -0.0050,  0.3479,  0.2972,  0.2676, -0.0133,  0.2610,  0.1894],
        [ 0.0913, -0.0061,  0.3264, -0.2588, -0.1708, -0.3519, -0.2913,  0.2256],
        [ 0.0130, -0.1475, -0.1744,  0.2541, -0.2094, -0.1143, -0.0697,

### 3.Positional Encoding Code

In [ ]:
import torch
import math

d_model = 6        # Each word represented by 6 numbers
max_len = 10       # Maximum sentence length (10 words)

pe = torch.zeros(max_len, d_model)  # Shape: [10, 6]   
# Position 1: [0, 0, 0, 0, 0, 0]
# Position 2: [0, 0, 0, 0, 0, 0]
# ...
# Position 10:[0, 0, 0, 0, 0, 0]

position = torch.arange(0, max_len).unsqueeze(1)
# torch.arange(0,10) → [0,1,2,3,4,5,6,7,8,9] a list
# unsqueeze(1) → [[0],[1],[2],...[9]]  Shape: [10,1]  
# unsqueeze(1)**: Reshapes the vector from a 1D array to a 2D column matrix (Shape: `[max_len, 1]`).

div_term=torch.exp(torch.arange(0,d_model,2)*-(math.log(10000)/d_model))

# PE(pos, 2i) = sin(pos / 10000^(2i / d_model))
# PE(pos, 2i+1) = cos(pos / 10000^(2i / d_model))

# 1/10000^(2i/d_model)  = 10000^(-2i/d_model)
                      # = exp(log(10000^(-2i/d_model)))
                      # = exp(-2i/d_model × log(10000))
                      # = exp(2i × (-log(10000)/d_model))
                      # 2i is exactly torch.arange(0, d_model, 2)  [0,2,4...]   when i=0, 2i=0 ;  i=1, 2i=2 .....
 
pe[:,0::2]=torch.sin(position*div_term)      # pe[:, 0::2] means: "Take all rows, and every 2nd column starting from column 0"
# position * div_term: [10,1] × [1,3] broadcasts to [10,3]
# sin() applied to each element

pe[:,1::2]=torch.cos(position*div_term)
# Fill odd columns (1,3,5) with cosine
# pe[:, 1::2] means: ALL rows (:), columns starting at 1, step 2 (1,3,5)
# Same position * div_term but with cos()
 
print("positional Encoding vector")
print(pe[0:5, :])                      
# Show first 5 rows (positions 0-4), all 6 columns
# pe[0:5, :] = rows 0 to 4, all columns
# Each position now has unique pattern: [sin, cos, sin, cos, sin, cos]

x = torch.rand(5,d_model)
# Random embeddings for 5 words, each 6-dim
# Shape: [5,6] - 5 words, each with 6 random features

x_pe = x + pe[:5]
# Add position info to first 5 words
# pe[:5] takes first 5 rows of positional encoding matrix
# Broadcasting: x[5,6] + pe[5,6] = [5,6]
# Each word's embedding now contains position information

print("\nEmbedding + Position")

print(x_pe)
         





positional Encoding vector
tensor([[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0464,  0.9989,  0.0022,  1.0000],
        [ 0.9093, -0.4161,  0.0927,  0.9957,  0.0043,  1.0000],
        [ 0.1411, -0.9900,  0.1388,  0.9903,  0.0065,  1.0000],
        [-0.7568, -0.6536,  0.1846,  0.9828,  0.0086,  1.0000]])

Embedding + Position
tensor([[ 0.5237,  1.3785,  0.6870,  1.8681,  0.1105,  1.6168],
        [ 1.0984,  1.5263,  0.0475,  1.4863,  0.0033,  1.2859],
        [ 1.0445,  0.0325,  0.1023,  1.9200,  0.4572,  1.6877],
        [ 0.9835, -0.2160,  0.5791,  1.9261,  0.1526,  1.5658],
        [-0.1558, -0.5278,  0.5460,  1.7794,  0.0693,  1.5467]])


### 4.Layer Normalization

In [4]:
import torch.nn as nn

layer_norm = nn.LayerNorm(6)

x = torch.rand(3,6)

print("Before LayerNorm")
print(x)

print("\nAfter LayerNorm")
print(layer_norm(x))

Before LayerNorm
tensor([[0.9233, 0.5711, 0.0938, 0.2498, 0.1352, 0.5575],
        [0.3563, 0.9915, 0.9294, 0.4082, 0.2791, 0.3336],
        [0.3311, 0.7295, 0.4909, 0.4838, 0.5177, 0.8602]])

After LayerNorm
tensor([[ 1.7175,  0.5114, -1.1232, -0.5890, -0.9815,  0.4648],
        [-0.6590,  1.5054,  1.2939, -0.4820, -0.9219, -0.7365],
        [-1.3606,  0.9191, -0.4463, -0.4869, -0.2926,  1.6673]],
       grad_fn=<NativeLayerNormBackward0>)


#### Before LayerNorm (Random Input):

tensor([[0.2, 0.5, 0.1, 0.8, 0.3, 0.6],   # Sample 1

        [0.7, 0.2, 0.9, 0.1, 0.4, 0.5],   # Sample 2
        
        [0.3, 0.6, 0.2, 0.7, 0.1, 0.8]])  # Sample 3

For Each Sample, LayerNorm:

**Step 1: Calculate mean and variance across the 6 features**

For Sample 1:

Mean = (0.2+0.5+0.1+0.8+0.3+0.6)/6 = 0.42

Variance = [ (0.2-0.42)² + (0.5-0.42)² + ... ]/6 = 0.06

Std = √0.06 = 0.24

**Step 2: Normalize each feature**

Feature 1: (0.2 - 0.42)/0.24 = -0.92

Feature 2: (0.5 - 0.42)/0.24 = 0.33

Feature 3: (0.1 - 0.42)/0.24 = -1.33


**Step 3: Scale and shift (learnable parameters γ, β)**

Output = γ × normalized + β

(Initially γ=1, β=0, so output = normalized)

#### After LayerNorm:

tensor([[-0.92,  0.33, -1.33,  1.58, -0.50,  0.83],   #Sample1

        [ 0.58, -0.83,  1.33, -1.33, -0.25,  0.50],   #Sample2

        [-0.58,  0.83, -0.92,  1.08, -1.33,  0.92]])  #Sample3

Each sample now has mean≈0 and variance≈1 across its 6 features.

#### Benefits of normalization:

1. Stable Training

Before: [0.1, 100, -5] → Next layer confused by huge range

After:  [-0.8, 1.5, -0.7] → Next layer sees consistent scale

No more adapting to wildly different input ranges!

2. Faster Convergence

Without LayerNorm: Loss decreases slowly 

With LayerNorm:    Loss decreases faster 

Network spends less time adjusting to scale, more time learning patterns!

3. Gradient Flow

Before: Large values → Saturated activation → Gradients vanish

After:  Normalized values → Activations sensitive → Gradients flow

Information flows better through the network!

### Encoder block

Input embedding

↓

add Positional embedding

↓

multi-head Self Attention

↓

Add & Norm

↓

Feed Forward

↓

Add & Norm

### 5.Encoder Code

In [9]:
import torch
import torch.nn as nn

# CLASS DEFINITION 
# A class is like a BLUEPRINT for creating objects
# It defines what properties and behaviors the object will have
class EncoderBlock(nn.Module):  # Inherits from nn.Module (PyTorch's base class)
    
    # CONSTRUCTOR METHOD
    # __init__ is the CONSTRUCTOR - it runs ONCE when creating a new object
    # It sets up all the LAYERS and PARAMETERS that this block will use
    def __init__(self, d_model):  # self = the object being created, d_model = parameter we pass
        super().__init__()  # Call parent class (nn.Module) constructor - REQUIRED for PyTorch
        
        # self.attn means: store this attention layer INSIDE the object
        # self = this object
        # .attn = variable name (like a label)
        # The layer becomes part of the object's data
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,    # Each word has d_model features
            num_heads=2,          # Use 2 parallel attention heads
            batch_first=True       # Input shape: [batch, seq_len, features]
        )
        
        # Store first layer normalization layer in the object
        self.norm1 = nn.LayerNorm(d_model)  # Normalizes across d_model features
        
        # Store feed-forward network in the object
        # Sequential = run these layers in order
        self.ff = nn.Sequential(
            nn.Linear(d_model, 2048),  # First linear: expand 8→32
            nn.ReLU(),                # Activation: adds non-linearity
            nn.Linear(2048, d_model)    # Second linear: compress 32→8
        )
        
        # Store second layer normalization in the object
        self.norm2 = nn.LayerNorm(d_model)
        
        # __init__ ends here - object is now created with all layers stored inside
    
    # ==================== FORWARD METHOD ====================
    # forward defines what happens when data PASSES THROUGH this block
    # It's called automatically when you do encoder(x)
    def forward(self, x):  # self = this object, x = input data
        
        # Step 1: Apply self-attention
        # self.attn accesses the attention layer stored inside the object
        # (x,x,x) means Query, Key, Value all come from same input (self-attention)
        attn_output, _ = self.attn(x, x, x)  # _ ignores attention weights
        
        # Step 2: Residual connection + LayerNorm
        # self.norm1 accesses first layer norm stored in object
        # x + attn_output = add original input to attention output (residual)
        x = self.norm1(x + attn_output)
        
        # Step 3: Feed-forward network
        # self.ff accesses the sequential network stored in object
        ff_output = self.ff(x)
        
        # Step 4: Another residual connection + LayerNorm
        # self.norm2 accesses second layer norm stored in object
        x = self.norm2(x + ff_output)
        
        return x  # Return processed data



# Create an INSTANCE (object) of the EncoderBlock class
# This RUNS __init__ and returns a complete object with all layers inside
encoder = EncoderBlock(8)  # 8 = d_model (feature dimension)

# Create some random input data
# 1 batch, 4 words, each with 8 features
x = torch.rand(1, 4, 8)    # Shape: [1,4,8]

# Pass data through the encoder object
# This AUTOMATICALLY CALLS the forward method
# Python sees 'encoder' is an object with __call__ method (from nn.Module)
# So encoder(x) becomes encoder.forward(x)
out = encoder(x)

print("Encoder Output")
print(out)

Encoder Output
tensor([[[-1.6053,  0.3275,  1.6247, -0.1278, -0.8968,  0.0898, -0.6211,
           1.2091],
         [-1.3299,  0.1266,  2.0240, -0.4582, -1.2757,  0.3484,  0.2700,
           0.2947],
         [-1.5524,  1.4313,  1.5009, -0.5651, -0.8531,  0.4039, -0.1065,
          -0.2591],
         [ 1.0431,  0.9708,  0.0167, -1.2228, -1.1162, -0.5629, -0.6894,
           1.5608]]], grad_fn=<NativeLayerNormBackward0>)


### Decoder Block

Masked Attention

↓

Add & Norm

↓

Cross Attention+multihead attention

↓

Add & Norm

↓

Feed forward

↓

Add & Norm

↓

Linear 

↓

Softmax

↓

Output probabilities

#### Mathematically order:

Embeddings

↓

Q,K,V

↓

QKᵀ

↓

Scaling

↓

Mask → -inf

↓

Softmax → 0 probability

↓

weights × V

↓

new embeddings(contexual embedding)

### 6.Masked head Attention Code

In [ ]:
# step-1 create embedding(hindi)

import torch
import torch.nn.functional as F
import math

# torch.nn = Classes with learnable parameters (stateful)
# F = Functions without parameters (stateless)
# nn.Linear has weights you train
# F.relu just applies activation function
# Both are needed for building neural networks!

torch.manual_seed(0)     # Locks the random number generator so you get the same random numbers every time you run the code.

seq_len = 3
d_model = 4

# embeddings for 4 words
x = torch.rand(seq_len, d_model)

print("Embeddings:")
print(x)

# Step-2 Create Q, K, V

Wq = torch.rand(d_model,d_model)
Wk = torch.rand(d_model,d_model)
Wv = torch.rand(d_model,d_model)

# nn.Linear = A fully connected layer that:
# Stores a learnable weight matrix and bias
# Computes output = input @ weight.T + bias
# Transforms input features into new features
# Fully Connected - Every input neuron connects to every output neuron with its own weight.

Q = x @ Wq
K = x @ Wk
V = x @ Wv

# Step-3 Compute Attention Scores

scores = Q @ K.T

print("\n Attention Scores before scaling :\n")
print(scores)

# Step-4 Apply Scaling

scores = scores / math.sqrt(d_model)

print("\nScaled Scores:\n")
print(scores)

# step-5 Create mask

# Create a 3×3 matrix of all ones
ones_matrix = torch.ones(seq_len, seq_len)
# Result:
# [[1, 1, 1],
#  [1, 1, 1],
#  [1, 1, 1],

# torch.tril() keeps only lower triangle (including diagonal), rest becomes 0
mask = torch.tril(torch.ones(seq_len, seq_len))
# Before (ones matrix):    After tril (lower triangular):
# [1, 1, 1]            [1, 0, 0]
# [1, 1, 1]     →      [1, 1, 0]
# [1, 1, 1]            [1, 1, 1]

print("\nMask matrix:\n")
print(mask)

# step-6 Apply mask

masked_scores = scores.masked_fill(mask == 0, float('-inf'))

# mask == 0 → Find positions where mask is 0 (future positions)
# float('-inf') → Replace those positions with negative infinity
# Keep other positions unchanged

print("\nScores After Masking:\n")
print(masked_scores)

# step-7 Softmax

weights = F.softmax(masked_scores, dim=-1)

# dim=0	Apply softmax down rows (each column sums to 1)	     Column-wise normalization
# dim=1	Apply softmax across columns (each row sums to 1)	 Row-wise normalization
# dim=-1	Last dimension (same as dim=1 for 2D)	         Row-wise normalization

print("\nAttention Weights After Softmax:\n")
print(weights)

# step-8 Final context vector

output = weights @ V

print("\nFinal Attention Output:\n")
print(output)





Embeddings:
tensor([[0.4963, 0.7682, 0.0885, 0.1320],
        [0.3074, 0.6341, 0.4901, 0.8964],
        [0.4556, 0.6323, 0.3489, 0.4017]])

 Attention Scores before scaling :

tensor([[1.1265, 2.1593, 1.6777],
        [2.0793, 4.1886, 3.1716],
        [1.5582, 3.0382, 2.3418]])

Scaled Scores:

tensor([[0.5633, 1.0797, 0.8388],
        [1.0396, 2.0943, 1.5858],
        [0.7791, 1.5191, 1.1709]])

Mask matrix:

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

Scores After Masking:

tensor([[0.5633,   -inf,   -inf],
        [1.0396, 2.0943,   -inf],
        [0.7791, 1.5191, 1.1709]])

Attention Weights After Softmax:

tensor([[1.0000, 0.0000, 0.0000],
        [0.2583, 0.7417, 0.0000],
        [0.2185, 0.4581, 0.3234]])

Final Attention Output:

tensor([[0.4227, 0.5555, 1.1286, 1.2097],
        [0.9199, 0.7064, 1.2686, 1.1977],
        [0.8263, 0.6819, 1.2521, 1.1864]])
